In [1]:
import warnings
warnings.filterwarnings('ignore')

(Cookbook_Saving_hbond_interactions)=
# Saving hydrogen-bond analyses

*Calculating, naming, persisting and querying attributed hydrogen bonds.*

Compare definitions without overwriting results or losing the calculation provenance.
This controlled water pair provides a small, reproducible pipeline with evaluated-empty
structures, named analyses and explicit source indices.

:::{versionadded} 1.0.0
:::

## Calculating named definitions

Each method produces an independent analysis. Attach it explicitly after calculation;
the MolSys domain validates compatible axes without authenticating independent origins.

Hydrogen-bond detection uses explicit donor–hydrogen pairs. A system without represented hydrogens can still have recognized acceptors while its calculated hydrogen-bond result is empty. The result retains evaluated structures and the examined participant scope; it does not reconstruct hydrogens. Check the input hydrogen inventory when interpreting an empty analysis.


In [2]:
import molsysmt as msm
import numpy as np
from rdkit import Chem
from molsysmt import pyunitwizard as puw

molsys = msm.convert(Chem.AddHs(Chem.MolFromSmiles('O.O')), to_form='molsysmt.MolSys')
xyz = np.array([[0, 0, 0], [.28, 0, 0], [.1, 0, 0], [0, .1, 0], [.28, .1, 0], [.28, 0, .1]])
coordinates = np.repeat(xyz[None], 3, axis=0)
coordinates[1, [1, 4, 5]] += [0, .4, 0]
molsys.structures.append(coordinates=puw.quantity(coordinates, 'nm'))

In [3]:
analyses = {method: msm.interactions.hbonds.get_hbonds(molsys, method=method, pbc=False)
            for method in ('baker_hubbard', 'cpptraj', 'prolif')}
molsys.interactions = {**molsys.interactions, **analyses}
print({name: result.n_interactions for name, result in molsys.interactions.items()})

{'baker_hubbard': 2, 'cpptraj': 2, 'prolif': 2}


## Persisting the system and provenance

Public conversion writes H5MSM 0.5. It retains method references, original producer
versions, measurement units, coverage and stable occurrence indices for this analysis
version. The installed reader version does not replace the saved producer version.

In [4]:
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory()
path = str(Path(workspace.name) / 'hydrogen_bonds.h5msm')
msm.convert(molsys, to_form=path)
restored = msm.convert(path, to_form='molsysmt.MolSys')
for name, result in analyses.items():
    loaded = restored.interactions[name]
    assert loaded.software == result.software
    assert loaded.parameters == result.parameters
    assert loaded.to_dict()['occurrence_indices'].tolist() == result.to_dict()['occurrence_indices'].tolist()
print(list(restored.interactions))

['baker_hubbard', 'cpptraj', 'prolif']


## Querying and remapping selected structures

Query the visible frame without recalculating. A frame evaluated without interactions
differs from one never evaluated. Extraction preserves requested order and remaps
local indices while retaining source maps. Removing a participant removes its relation.

In [5]:
analysis = restored.interactions['baker_hubbard']
print(analysis.query(structure_indices=[2, 0, 2], atom_indices=[2]).to_dict()['occurrence_indices'])
assert analysis.query(structure_indices=1).n_interactions == 0
subset = msm.extract(restored, structure_indices=[2, 0])
print(subset.interactions['baker_hubbard'].structure_source_indices.tolist())
without_acceptor = msm.extract(restored, selection=[0, 2, 3])
assert without_acceptor.interactions['baker_hubbard'].n_interactions == 0

[1 0]
[2, 0]


## Calculating directly from file blocks

Index selections read chemical information once and only projected coordinate blocks.
Saved analyses are not required for a fresh calculation; accepted output remains in RAM.

In [6]:
with msm.configure.context(chunk_size=1):
    projected = msm.interactions.hbonds.get_hbonds(
        path, structure_indices=[2, 0, 2], heavy_mode='force', pbc=False)
print(projected.execution_records[0]['details']['execution_chunks'], projected.occurrence_structures.tolist())
assert projected.evaluated_structure_indices.tolist() == [0, 2]
workspace.cleanup()

2 [0, 2]


## Deriving directions from declared donor sites

Recognize fixed-state donor-H pairs before requesting their observed geometry. The site inventory retains chemical method/state evidence; the vector dictionary retains geometry, axes and producer version. Keep both records together in a derived site inventory. This calculation leaves the molecular system and its named interaction analyses unchanged.

Acceptor indices alone do not determine lone-pair directions. This recipe supplies donor-H geometry only, using explicit indexed hydrogens.

In [7]:
sites = msm.physchem.get_hbond_sites(molsys, method='smarts_donor_acceptor', structure_indices=[2, 0])
donor_geometry = msm.structure.get_vectors(
    molsys, selection=sites['donor_hydrogen_pairs'], pairs=True,
    structure_indices=[2, 0], pbc=False, output_type='dictionary', heavy_mode='force')
assert donor_geometry['structure_indices'].tolist() == [2, 0]
assert donor_geometry['directions'].shape[1] == len(sites['donor_hydrogen_pairs'])
print(donor_geometry['directions'].shape)

(2, 4, 3)


:::{seealso}
:class: dropdown

- {ref}`Tutorial_Get_hbonds` — scientific definitions, scopes, units and images.
- {ref}`Tutorial_Get_hbond_sites` — reusable chemical-site recognition.
- {ref}`user-foundations-native-world-classes-molsysmt-molsys` — named-domain ownership.
:::